# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR^2 colorectal cancer survivors dataset using the `mlcroissant` library, referencing all dataset elements by their `@id` fields.

### Dataset Source
The dataset is defined by a Croissant schema accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the FAIR^2 dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset from metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset overview
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
List available record sets (and their `@id`s) in the dataset, and display their available fields (`@id` only).

*Note: All references to record sets and fields will use their Croissant `@id`.*

In [ ]:
# List all record sets and fields, referencing `@id`
record_sets = list(dataset.record_sets())
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    field_ids = []
    fields = rs.get('field', [])
    if isinstance(fields, dict):  # single field
        field_ids = [fields['@id']]
    elif isinstance(fields, list):
        field_ids = [f['@id'] for f in fields if isinstance(f, dict) and '@id' in f]
    print(f"  Fields: {field_ids}\n")

Below, we iterate through a sample of the records in the primary record set (referenced by its `@id`). This helps us understand the structure and sample data.

In [ ]:
# Display sample records from the main tabular record set using its @id
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]['@id']  # Use the first record set here for demonstration
    print(f"\nDisplaying a few sample records from record set: {main_record_set_id}\n")
    for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
        print(rec)
        if i >= 2:
            break
else:
    print('No record sets found in the dataset.')

## 3. Data Extraction
Load data from the relevant record set(s) into DataFrames, referencing record set and field `@id`s. For the FAIR^2 dataset, there is one main table of records.

*Field and column names are always handled as `@id`s for downstream operations.*

In [ ]:
# Prepare mapping of record set @ids and load each as a DataFrame
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for record_set_id in record_set_ids:
    records_list = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records_list)
    dataframes[record_set_id] = df
    print(f'Record set {record_set_id}: {len(df)} records loaded.')

# For demonstration, list columns of the main record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nMain record set columns (@id):\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Now, we will process and analyze the data. Example tasks: filter records by numeric `@id` column, normalize a field, and group by another field, **referencing all as `@id` values**.

In [ ]:
# For demonstration, choose two field @ids as numeric and grouping keys.
# You may adjust these after inspecting the field list above.
main_df = dataframes[main_record_set_id]
columns = main_df.columns.tolist()

# Example: Suppose @id of numeric field is 'Age' and grouping field is 'Sex' (use real @id from your schema)
numeric_field_id = None
group_field_id = None

# Heuristically pick based on expected column names
for col in columns:
    if 'age' in col.lower():
        numeric_field_id = col
    elif 'sex' in col.lower() or 'gender' in col.lower():
        group_field_id = col

if numeric_field_id is None or group_field_id is None:
    print('Could not identify numeric_field_id or group_field_id by name heuristics. Please inspect columns and set manually.')

# Proceed only if columns found
if numeric_field_id and group_field_id:
    threshold = 60
    filtered_df = main_df[pd.to_numeric(main_df[numeric_field_id], errors='coerce') > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (referenced by @id):")
    display(filtered_df[[numeric_field_id, group_field_id]].head())

    # Normalize the numeric field
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field_id}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') -
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
    
    print(f"\nNormalized {numeric_field_id}:")
    display(
        filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized", group_field_id]].head()
    )

    # Group by group_field_id and show means
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name='mean_'+numeric_field_id)
        print(f"\nMean {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
else:
    print('Numeric or group field not found in main record set. Please review columns and adjust field @ids as appropriate.')

## 5. Visualization
Visualize the normalized distribution of the numeric column, split by the grouping field, referencing column names as their `@id`.

*Adjust field @ids if necessary for your data schema.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if numeric_field_id and group_field_id and f"{numeric_field_id}_normalized" in filtered_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df, x=f"{numeric_field_id}_normalized", hue=group_field_id, kde=True, palette='muted', bins=15)
    plt.title(f"Distribution of Normalized {numeric_field_id} by {group_field_id}")
    plt.xlabel(f"{numeric_field_id} (normalized)")
    plt.ylabel("Count")
    plt.legend(title=group_field_id)
    plt.tight_layout()
    plt.show()
else:
    print('Visualization skipped: Suitable numeric and grouping fields not found or not processed.')

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and process a biomedical tabular dataset using the `mlcroissant` library, referencing schema elements consistently by their Croissant `@id` values. We:
- Loaded dataset metadata and records
- Explored available record sets and fields
- Loaded tabular data and accessed fields by their unique `@id`s
- Performed basic EDA (filtering, normalization, grouping)
- Visualized results referencing the underlying schema

**Next Steps:**
- Adapt field and grouping `@id` values as necessary for your data schema
- Expand analysis or modeling code as required for your scientific objective.